# TP7 : ajouter un broker, réassigner, redémarrer sans interruption, décommissionner

| Repère | Valeur |
|---|---|
| Moment | Jour 2, 13:40-13:55 : étapes 6.1 et 6.3 jouées par le formateur (démo exploitation) ; TP complet en autonomie après la formation |
| Compétences | C14 Réassigner des partitions, ajouter et retirer un broker, ajouter un log dir ; C15 Rolling restart sans interruption de service |
| Slides de rappel | S61 (log dirs), S64 (configuration statique et dynamique), S65 (réassignation), S66 (ajout), S67 (décommissionnement), S68 (rolling restart) |
| Version Markdown | `enonce.md`, mêmes étapes en ligne de commande seule |

**Mode d'emploi du notebook**

- Exécutez les cellules dans l'ordre avec `Maj+Entrée`. Après un redémarrage du noyau, les variables sont perdues : relancez les trois cellules du début (connexion, fonctions d'aide, outils du TP), puis les cellules « Code fourni ». Ne relancez pas une cellule qui modifie le cluster (création d'un topic, réassignation, redémarrage d'un nœud) : elle échouerait, ou referait l'action.
- Chaque étape suit le même schéma : une explication, un **exemple en ligne de commande**, une cellule **`# TODO`**, puis une **vérification**. Les commandes qui modifient le cluster sont affichées sans être exécutées (« ne pas lancer en plus du TODO »).
- **Vous n'avez pas de code à écrire.** Dans une cellule `# TODO`, remplacez chaque `...` par une valeur : un **réglage** (une liste de brokers, un débit...) ou votre **prévision** du résultat (un nombre, `True` ou `False`). Les noms qui commencent par `PREDICTED_` sont des prévisions : décidez-les avant d'exécuter, la vérification les compare au résultat.
- Sous les réponses, le code est fourni : lisez ses commentaires, ne le modifiez pas. Tant qu'il reste un `...`, la ligne `if filled(...):` l'empêche de s'exécuter et affiche « À compléter ».
- Le client Python n'a pas d'API pour la réassignation, les répertoires de données (log dirs), le quorum ni le désenregistrement d'un broker. Dans ce TP, le code fourni **pilote les outils Java** par des cellules `!docker exec kafka-tools ...` et **exploite leurs sorties** : les plans de réassignation et l'état des log dirs sont en JSON.
- Les plans sont écrits dans `/tmp` du conteneur `tools`, comme dans la version Markdown.

In [ ]:
# Connexion au lab : adresses des brokers, configuration du client Python, dossier de travail
import os
import time

from confluent_kafka.admin import AdminClient

BS = os.environ["BS"]  # listener INTERNAL, comme $BS dans le conteneur tools
CONF = {"bootstrap.servers": BS, "log_level": 2}

# Les commandes docker compose et les fichiers config/ se lisent depuis le dossier lab/
LAB_DIR = os.environ["FORMATION_DIR"] + "/lab"
os.chdir(LAB_DIR)
os.environ["COMPOSE_PROGRESS"] = "plain"  # docker compose sans animation dans le notebook

In [ ]:
# Trois fonctions d'aide, écrites ici : elles ne font pas partie de confluent-kafka
def wait_for(condition, timeout=60, step=2, message="condition"):
    """Rappelle condition() jusqu'à obtenir une valeur vraie, et la renvoie."""
    start = time.time()
    while True:
        try:
            value = condition()
        except Exception:
            value = None
        if value:
            return value
        if time.time() - start >= timeout:
            raise TimeoutError("%s : non atteinte en %d s" % (message, timeout))
        time.sleep(step)

def table(lines, headers):
    """Affiche des lignes en tableau Markdown dans le notebook."""
    from IPython.display import Markdown, display

    md = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
    md += ["| " + " | ".join(str(v).replace("|", "\\|") for v in line) + " |" for line in lines]
    display(Markdown("\n".join(md)))

def filled(*answers):
    """Vrai si toutes les réponses d'une cellule TODO sont données ; sinon affiche « À compléter »."""
    for answer in answers:
        if answer is ...:
            print("À compléter : remplacez chaque ... de la cellule par votre réponse, puis exécutez-la de nouveau.")
            return False
    return True

In [ ]:
# Outils du TP : imports, et fonctions de lecture de l'état du cluster
import json
import os
import shutil
import threading
import time
from collections import Counter

from confluent_kafka import ElectionType, KafkaException, Producer
from confluent_kafka.admin import AlterConfigOpType, ConfigEntry, ConfigResource, ConfigSource, NewTopic


def placement(topics=None):
    """{(topic, partition): (leader, [replicas])}, pour les topics donnés ou pour tous les topics."""
    admin = AdminClient(CONF)
    meta = admin.list_topics(timeout=10).topics
    return {(t, p.id): (p.leader, p.replicas) for t in (topics or meta) for p in meta[t].partitions.values()}


def replicas_per_broker(topics=None):
    """Nombre de répliques hébergées par chaque broker : {broker: nombre}."""
    return dict(sorted(Counter(b for _, replicas in placement(topics).values() for b in replicas).items()))


def urp():
    """Nombre de partitions sous-répliquées (moins de répliques dans l'ISR que de répliques prévues)."""
    admin = AdminClient(CONF)
    return sum(1 for t in admin.list_topics(timeout=10).topics.values()
               for p in t.partitions.values() if len(p.isrs) < len(p.replicas))


def log_dirs(broker, topics=None):
    """{répertoire: {partition: taille}} d'un broker, lu dans la sortie JSON de kafka-log-dirs.sh."""
    options = " --topic-list " + ",".join(topics) if topics else ""
    # La dernière ligne de la sortie est le JSON
    output = !docker exec kafka-tools bash -c "kafka-log-dirs.sh --bootstrap-server $BS --describe --broker-list {broker}{options} | tail -1" 2>&1
    output = output.n
    return {d["logDir"]: {p["partition"]: p["size"] for p in d["partitions"]}
            for b in json.loads(output)["brokers"] for d in b["logDirs"]}


def healthy(node):
    """Vrai si Docker voit le conteneur kafka-node sain (son contrôle de santé répond)."""
    output = !docker compose ps --format '{{{{.Health}}}}' kafka-{node} 2>&1
    return output.n == "healthy"


admin = AdminClient(CONF)
print("Brokers :", sorted(n.id for n in admin.describe_cluster().result().nodes), "- URP :", urp())

## 1. Objectifs

À la fin du TP, vous savez :

- ajouter un broker à un cluster KRaft et vérifier son enregistrement ;
- produire, exécuter et suivre un plan de réassignation, avec une limite de débit ;
- rééquilibrer les leaders après une réassignation ;
- ajouter un répertoire de données à un nœud et y déplacer une réplique ;
- réaliser un rolling restart sous charge, sans perte ni échec côté producteur ;
- retirer un broker proprement : cordon, réassignation, arrêt, désenregistrement.

## 2. Prérequis

- Lab en mode `extended`, sain : `scripts/validate.sh`. Lab démarré par `scripts/start.sh` (sinon `docker compose` ne peut pas créer `kafka-4`).
- Topics `orders` (6 partitions) et `payments` (3 partitions) présents, issus des TP2 et TP4.
- Aucune partition sous-répliquée.
- Notions : slides S61 à S68.

## 3. Architecture utilisée

```
  Avant                                  Pendant le TP                         Après 6.7
  kafka-1  broker+contrôleur  az-a       kafka-1  data + data2 (6.5)            kafka-1  data + data2
  kafka-2  broker+contrôleur  az-b       kafka-2                                kafka-2
  kafka-3  broker+contrôleur  az-c       kafka-3                                kafka-3
                                         kafka-4  broker seul, az-a             (retiré, désenregistré)
                                                  observateur du quorum
```

| Nœud | Rôles | Rack | Remarque |
|---|---|---|---|
| kafka-1 à kafka-3 | broker, controller (votants) | az-a, az-b, az-c | Quorum de 3 votants |
| kafka-4 | broker seul | az-a | Profil Compose `scale` ; formaté avec `--no-initial-controllers` |

| Lab | Production |
|---|---|
| Nœuds combinés broker et contrôleur | Contrôleurs dédiés (3 ou 5), sur leurs propres machines |
| Le nouveau répertoire est formaté par le script de démarrage (`format --ignore-formatted`) | Formatage du disque ajouté : étape manuelle de la procédure, avant le redémarrage |
| Plans générés par `--generate`, qui ne tient compte que du nombre de répliques | Outil de rééquilibrage qui tient compte de la charge (Cruise Control, par exemple) |

## 4. Contexte métier

Le Black Friday approche. NovaShop ajoute un quatrième broker et doit y répartir les topics critiques sans interrompre les ventes. Le disque de `kafka-1` est presque plein : un second disque est installé. Un correctif de sécurité impose ensuite un redémarrage de tous les nœuds en pleine journée. Enfin, après l'événement, le broker temporaire est rendu.

## 5. Énoncé

1. Démarrez `kafka-4` et prouvez qu'il fait partie du cluster.
2. Montrez qu'il ne reçoit aucune partition existante.
3. Répartissez `orders-history`, puis `orders` et `payments`, sur les 4 brokers, avec une limite de débit.
4. Ajoutez le répertoire `/var/lib/kafka/data2` à `kafka-1` et déplacez-y une réplique de `payments`.
5. Redémarrez les 3 nœuds d'origine un par un, sous un producteur `acks=all`, sans aucun envoi en échec.
6. Retirez `kafka-4` du cluster sans perdre de réplique.

## 6. Étapes

### 6.1 Démarrer kafka-4

**Exemple en ligne de commande** : les paramètres clés du nouveau nœud.

In [ ]:
!grep -E '^(process.roles|node.id|controller.quorum.bootstrap.servers|broker.rack|log.dirs)=' config/kafka/kafka-4/server.properties

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [hôte]
docker compose --profile scale up -d kafka-4
```

In [ ]:
# TODO 6.1 : remplacez les deux ... puis exécutez
PREDICTED_RACK = ...         # rack de kafka-4, lu dans broker.rack ci-dessus (un texte)
PREDICTED_QUORUM_ROLE = ...  # rôle de kafka-4 (process.roles=broker) dans le quorum :
                             # "Leader", "Follower" ou "Observer" ?

# Code fourni, à lire : rien à modifier en dessous
brokers = {}
startup_logs = ""
if filled(PREDICTED_RACK, PREDICTED_QUORUM_ROLE):
    admin = AdminClient(CONF)
    !docker compose --profile scale up -d kafka-4
    # Attente : le broker 4 apparaît parmi les brokers enregistrés
    wait_for(lambda: 4 in [n.id for n in admin.describe_cluster().result().nodes], timeout=90,
             message="enregistrement du broker 4")
    # {identifiant: rack} de chaque broker
    brokers = {n.id: n.rack for n in sorted(admin.describe_cluster().result().nodes, key=lambda n: n.id)}
    # Les traces de démarrage de kafka-4 : formatage du stockage, puis démarrage
    startup_logs = !docker compose logs kafka-4 | grep -E 'start-node|Formatting|Kafka Server started' 2>&1
    startup_logs = startup_logs.n

print(brokers)

In [ ]:
# Vérification 6.1
assert brokers == {1: "az-a", 2: "az-b", 3: "az-c", 4: "az-a"}, "À revoir : 4 brokers, kafka-4 dans le rack az-a"
print("OK  4 brokers, kafka-4 dans le rack az-a")
assert "Formatting metadata directory" in startup_logs and "Kafka Server started" in startup_logs, "À revoir : kafka-4 formaté puis démarré"
print("OK  kafka-4 formaté puis démarré")
assert PREDICTED_RACK == brokers[4], "À revoir : prévision %r, rack %s" % (PREDICTED_RACK, brokers[4])
print("OK  prévision juste : rack %s" % brokers[4])
# Statut du nœud 4 : dernière colonne de sa ligne dans describe --replication
quorum = !docker exec kafka-tools kafka-metadata-quorum.sh --bootstrap-server $BS describe --replication 2>&1
role_4 = [line.split()[-1] for line in quorum if line.split()[:1] == ["4"]]
assert role_4 == [PREDICTED_QUORUM_ROLE], "À revoir : prévision %r, statut %s" % (PREDICTED_QUORUM_ROLE, role_4)
print("OK  prévision juste : %s" % PREDICTED_QUORUM_ROLE)

Statut dans le quorum et contenu du répertoire de données de `kafka-4` (lecture seule) :

In [ ]:
!docker exec kafka-tools kafka-metadata-quorum.sh --bootstrap-server $BS describe --replication
!docker exec kafka-tools kafka-log-dirs.sh --bootstrap-server $BS --describe --broker-list 4 | tail -1

**Résultat attendu** :

- `process.roles=broker`, `node.id=4`, `controller.quorum.bootstrap.servers` identique aux autres nœuds, `broker.rack=az-a` ;
- traces : `[start-node] Formatage du stockage (cluster W5FmqobtRp-BMcXlZhVKBw)`, `Formatting metadata directory /var/lib/kafka/data with metadata.version 4.3-IV0.`, puis `Kafka Server started` ;
- 4 brokers ; dans le quorum, 3 lignes `Leader` ou `Follower`, et une ligne `4 ... Observer` avec un lag de 0 ;
- répertoire de `kafka-4` vide : `"logDir":"/var/lib/kafka/data"` avec `"partitions":[]`.

### 6.2 Constater la répartition

Les fonctions `placement()` et `replicas_per_broker()` de la cellule d'outils lisent le placement des répliques.

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-topics.sh --bootstrap-server $BS --create --topic orders-history --partitions 4 --replication-factor 3
kafka-producer-perf-test.sh --bootstrap-server $BS --topic orders-history \
  --num-records 20000 --record-size 500 --throughput -1
```

In [ ]:
# TODO 6.2 : remplacez les deux ... par vos prévisions, puis exécutez
PREDICTED_EXISTING_ON_4 = ...  # combien de répliques d'orders et de payments le broker 4 héberge-t-il,
                               # avant toute réassignation ? (un nombre)
PREDICTED_NEW_TOPIC_ON_4 = ...  # le nouveau topic orders-history aura-t-il des répliques sur le broker 4 ?
                                # (True ou False)

# Code fourni, à lire : rien à modifier en dessous
before = history = {}
if filled(PREDICTED_EXISTING_ON_4, PREDICTED_NEW_TOPIC_ON_4):
    admin = AdminClient(CONF)
    before = replicas_per_broker(["orders", "payments"])  # répliques par broker, topics existants

    # Nouveau topic, et 20000 messages de 500 octets (seulement à la première exécution)
    if "orders-history" not in admin.list_topics(timeout=10).topics:
        admin.create_topics([NewTopic("orders-history", num_partitions=4, replication_factor=3)])["orders-history"].result()
        producer = Producer({"bootstrap.servers": BS, "log_level": 4})
        for _ in range(20000):
            while True:
                try:
                    producer.produce("orders-history", value=b"x" * 500)
                    break
                except BufferError:
                    producer.poll(0.1)  # file locale pleine : laisser partir des messages
            producer.poll(0)
        producer.flush(60)
    history = replicas_per_broker(["orders-history"])  # répliques par broker, nouveau topic

print("orders et payments :", before, "- orders-history :", history)

In [ ]:
# Vérification 6.2
assert 4 not in before, "À revoir : aucune réplique existante sur le broker 4"
print("OK  aucune réplique existante sur le broker 4")
assert history.get(4, 0) > 0, "À revoir : le nouveau topic utilise le broker 4 : %s" % history
print("OK  le nouveau topic utilise le broker 4 : %s" % history)
assert PREDICTED_EXISTING_ON_4 == before.get(4, 0), "À revoir : prévision %r, %d réplique(s) sur le broker 4" % (PREDICTED_EXISTING_ON_4, before.get(4, 0))
print("OK  prévision juste : aucune réplique existante sur le broker 4")
assert PREDICTED_NEW_TOPIC_ON_4 is True, "À revoir : prévision %r : orders-history a %d répliques sur le broker 4" % (PREDICTED_NEW_TOPIC_ON_4, history.get(4, 0))
print("OK  prévision juste : %d répliques d'orders-history sur le broker 4" % history[4])

**Résultat attendu** : aucune réplique d'`orders` ni de `payments` sur le broker 4 ; `orders-history` en a plusieurs. Un nouveau topic utilise tous les brokers disponibles, les anciens ne bougent pas.

### 6.3 Premier cycle de réassignation

`kafka-reassign-partitions.sh --generate` propose un plan à partir d'une liste de topics et de brokers. Sa sortie contient deux lignes JSON, chacune après un titre : `Current partition replica assignment` (la répartition actuelle, votre retour arrière) et `Proposed partition reassignment configuration` (le plan proposé).

**Code fourni** : `execute(plan, rate)` écrit le plan dans `/tmp/plan.json` du conteneur `tools` et lance `--execute` (avec `--throttle` si un débit est donné) ; `verify_plan()` lance `--verify` sur ce fichier. La cellule suivante en ajoute quatre autres.

**Exemple en ligne de commande** : un plan généré pour `orders-history` (lecture seule : `--generate` ne change rien).

In [ ]:
# Code fourni : lancer et suivre un plan de réassignation
def execute(plan, rate=None):
    """Écrit le plan dans /tmp/plan.json (conteneur tools), puis lance --execute ; renvoie la sortie."""
    !echo '{json.dumps(plan)}' | docker exec -i kafka-tools tee /tmp/plan.json > /dev/null
    option = " --throttle %d" % rate if rate else ""  # limite de débit de la copie, en octets par seconde
    output = !docker exec kafka-tools bash -c "kafka-reassign-partitions.sh --bootstrap-server $BS --reassignment-json-file /tmp/plan.json --execute{option}" 2>&1
    return output.n


def verify_plan():
    """Lance --verify sur /tmp/plan.json : état de chaque partition, et retrait des limitations à la fin."""
    output = !docker exec kafka-tools kafka-reassign-partitions.sh --bootstrap-server $BS --reassignment-json-file /tmp/plan.json --verify 2>&1
    return output.n


!echo '{{"version":1,"topics":[{{"topic":"orders-history"}}]}}' | docker exec -i kafka-tools tee /tmp/topics.json
!docker exec kafka-tools bash -c "kafka-reassign-partitions.sh --bootstrap-server $BS --topics-to-move-json-file /tmp/topics.json --broker-list 1,2,3,4 --generate | cut -c1-200"

In [ ]:
# Code fourni : générer un plan, lire les limitations, attendre la fin, compter les leaders non préférés
def generate(topics, brokers):
    """Lance --generate pour ces topics et ces brokers ; renvoie (répartition actuelle, plan proposé)."""
    # 1. La liste des topics, au format attendu par l'outil, dans /tmp/topics.json du conteneur tools
    topics_json = json.dumps({"version": 1, "topics": [{"topic": t} for t in topics]})
    !echo '{topics_json}' | docker exec -i kafka-tools tee /tmp/topics.json > /dev/null
    # 2. --generate avec la liste des brokers cibles : "1,2,3,4"
    addresses = ",".join(str(b) for b in brokers)
    output = !docker exec kafka-tools bash -c "kafka-reassign-partitions.sh --bootstrap-server $BS --topics-to-move-json-file /tmp/topics.json --broker-list {addresses} --generate" 2>&1
    output = output.n
    # 3. Chaque JSON est sur la ligne qui suit son titre
    lines = output.splitlines()
    current = json.loads(lines[lines.index("Current partition replica assignment") + 1])
    proposed = json.loads(lines[lines.index("Proposed partition reassignment configuration") + 1])
    return current, proposed


def broker_throttles(broker):
    """{nom: valeur} des limitations de réplication posées dynamiquement sur un broker."""
    admin = AdminClient(CONF)
    future = list(admin.describe_configs([ConfigResource("broker", str(broker))]).values())[0]
    return {name: e.value for name, e in future.result().items()
            if "throttled" in name and e.source == ConfigSource.DYNAMIC_BROKER_CONFIG.value}


def wait_until_done(timeout):
    """Relance --verify toutes les 5 s jusqu'à la fin de la réassignation ; renvoie la dernière sortie."""
    def done():
        output = verify_plan()
        # Dernière ligne écrite par --verify quand tout est terminé
        return output if "Clearing topic-level throttles" in output else None
    return wait_for(done, timeout=timeout, step=5, message="fin de la réassignation")


def non_preferred_leaders(topics):
    """Nombre de partitions dont le leader n'est pas le premier de la liste des répliques (le préféré)."""
    return sum(1 for leader, replicas in placement(topics).values() if leader != replicas[0])

In [ ]:
# TODO 6.3 a : remplacez les deux ... puis exécutez
BROKERS = ...                 # brokers cibles du plan pour orders-history : [1, 2, 3] ou [1, 2, 3, 4] ?
PREDICTED_THROTTLE_SET = ...  # juste après --execute avec --throttle, le broker 4 porte-t-il une limitation
                              # leader.replication.throttled.rate ? (True ou False)

# Code fourni, à lire : rien à modifier en dessous
rollback = plan = {}
throttles_4 = {}
if filled(BROKERS, PREDICTED_THROTTLE_SET):
    rollback, plan = generate(["orders-history"], BROKERS)  # rollback : la répartition actuelle, à garder
    print(execute(plan, 50000000))                          # copie limitée à 50 Mo/s
    throttles_4 = broker_throttles(4)                       # lues aussitôt, pendant la copie

print("partitions du plan :", len(plan.get("partitions", [])), "- limitations sur le broker 4 :", throttles_4)

In [ ]:
# Vérification 6.3 a
assert len(plan.get("partitions", [])) == 4 and len(rollback.get("partitions", [])) == 4, "À revoir : plan et retour arrière pour les 4 partitions"
print("OK  plan et retour arrière pour les 4 partitions")
assert throttles_4.get("leader.replication.throttled.rate") == "50000000", "À revoir : limitation posée sur le broker 4"
print("OK  limitation posée sur le broker 4")
assert sorted(BROKERS) == [1, 2, 3, 4], "À revoir : réponse %r : le plan doit inclure le broker 4" % (BROKERS,)
print("OK  plan sur les brokers 1 à 4")
assert PREDICTED_THROTTLE_SET == ("leader.replication.throttled.rate" in throttles_4), "À revoir : prévision %r : limitations du broker 4 : %s" % (PREDICTED_THROTTLE_SET, throttles_4)
print("OK  prévision juste : limitation posée pendant la copie")

Équivalent en ligne de commande du `TODO` b (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-reassign-partitions.sh --bootstrap-server $BS --reassignment-json-file /tmp/plan.json --verify
kafka-leader-election.sh --bootstrap-server $BS --election-type preferred --all-topic-partitions
```

In [ ]:
# TODO 6.3 b : remplacez les deux ... puis exécutez
PREDICTED_THROTTLES_AFTER = ...  # une fois la réassignation terminée et --verify relancé, reste-t-il des
                                 # limitations sur le broker 4 ? (True ou False)
ELECTION_TYPE = ...              # élection qui remet en tête les leaders préférés :
                                 # ElectionType.PREFERRED ou ElectionType.UNCLEAN ?

# Code fourni, à lire : rien à modifier en dessous
end_73 = ""
throttles_after = None
non_preferred = non_preferred_after = None
if filled(PREDICTED_THROTTLES_AFTER, ELECTION_TYPE):
    admin = AdminClient(CONF)
    end_73 = wait_until_done(120)  # --verify toutes les 5 s, jusqu'à la fin
    print(end_73)
    throttles_after = broker_throttles(4)

    # Leaders non préférés, avant puis après l'élection
    non_preferred = non_preferred_leaders(["orders-history"])
    admin.elect_leaders(ELECTION_TYPE, None).result()  # None : toutes les partitions
    non_preferred_after = non_preferred_leaders(["orders-history"])

print("leaders non préférés avant l'élection :", non_preferred, "- après :", non_preferred_after)

In [ ]:
# Vérification 6.3 b
assert "Clearing broker-level throttles" in end_73, "À revoir : --verify a retiré les limitations"
print("OK  --verify a retiré les limitations")
assert throttles_after == {}, "À revoir : plus aucune limitation sur le broker 4"
print("OK  plus aucune limitation sur le broker 4")
assert non_preferred_after == 0, "À revoir : chaque partition d'orders-history a son leader préféré"
print("OK  chaque partition d'orders-history a son leader préféré")
assert PREDICTED_THROTTLES_AFTER is False, "À revoir : prévision %r : --verify retire les limitations" % (PREDICTED_THROTTLES_AFTER,)
print("OK  prévision juste : limitations retirées")

**Résultat attendu** :

- le plan proposé utilise `"log_dirs":["any","any","any"]` ; `--execute` répond `The inter-broker throttle limit was set to 50000000 B/s` ;
- juste après : `leader.replication.throttled.rate` et `follower.replication.throttled.rate` à 50000000 sur le broker 4 ;
- `--verify` : `Reassignment of partition orders-history-N is completed.` pour les 4 partitions, puis `Clearing broker-level throttles on brokers 1,2,3,4` et `Clearing topic-level throttles on topic orders-history` ;
- après réassignation, certains leaders ne sont plus le premier de leur liste ; l'élection les remet en tête.

### 6.4 Étendre `orders` et `payments` à kafka-4

Même cycle, avec les fonctions fournies en 6.3, et une limite de 1 Mo/s. La fonction `log_dirs(broker, topics)` de la cellule d'outils lit la taille des répliques d'un broker.

In [ ]:
# TODO 6.4 : remplacez les deux ... puis exécutez UNE SEULE FOIS
RATE_74 = ...             # limite de débit de la copie, 1 Mo/s, en octets par seconde (un nombre)
PREDICTED_BALANCED = ...  # après la réassignation, les 4 brokers auront-ils le même nombre de répliques
                          # d'orders et de payments ? (True ou False ; pensez aux racks)

# Code fourni, à lire : rien à modifier en dessous
first_verify = ""
on_4 = distribution_74 = {}
if filled(RATE_74, PREDICTED_BALANCED):
    if RATE_74 not in (1000000, 1048576):
        # Protection : une limite trop basse rendrait la copie interminable
        print("Valeur refusée : 1 Mo/s font 1000000 octets par seconde (ou 1048576 en unités binaires).")
    else:
        rollback_74, plan_74 = generate(["orders", "payments"], [1, 2, 3, 4])
        print(execute(plan_74, RATE_74))
        first_verify = verify_plan()  # état de la copie aussitôt après le lancement
        print(first_verify)
        wait_until_done(180)
        on_4 = log_dirs(4, ["orders", "payments"])                  # répliques du broker 4 et leur taille
        distribution_74 = replicas_per_broker(["orders", "payments"])  # répliques par broker

print("sur le broker 4 :", on_4)
print("répartition :", distribution_74)

In [ ]:
# Vérification 6.4
assert "Status of partition reassignment" in first_verify, "À revoir : suivi lancé aussitôt après --execute"
print("OK  suivi lancé aussitôt après --execute")
assert distribution_74.get(4, 0) > 0 and broker_throttles(4) == {}, "À revoir : répliques sur le broker 4, limitations retirées"
print("OK  répliques sur le broker 4, limitations retirées")
balanced = len(set(distribution_74.values())) == 1
assert PREDICTED_BALANCED == balanced, "À revoir : prévision %r, répartition %s" % (PREDICTED_BALANCED, distribution_74)
print("OK  prévision juste : répartition %s" % distribution_74)

**Résultat attendu** : `The inter-broker throttle limit was set to 1000000 B/s`. Selon le volume des partitions, le premier `--verify` montre des partitions `still in progress` ; avec des partitions très petites, tout peut être `completed` dès le premier `--verify`. Le broker 4 héberge ensuite des répliques des deux topics, avec leur taille en octets.

### 6.5 Défi : ajouter un log dir

Le paramètre `log.dirs` de `config/kafka/kafka-1/server.properties` liste les répertoires de données du nœud, séparés par des virgules. Le fichier est lisible et modifiable depuis le notebook (`KAFKA_1_FILE`). C'est une configuration **statique** : le nœud doit redémarrer pour la prendre en compte.

Un plan de réassignation peut aussi déplacer une réplique **entre les répertoires d'un même broker** : `log_dirs` contient un élément par réplique, dans le même ordre que `replicas`, et la valeur `"any"` laisse le broker choisir.

```json
{"version":1,"partitions":[{"topic":"payments","partition":<n>,"replicas":[<répliques actuelles, dans l'ordre>],"log_dirs":[<un élément par réplique>]}]}
```

In [ ]:
KAFKA_1_FILE = os.path.join(LAB_DIR, "config/kafka/kafka-1/server.properties")
!grep '^log.dirs' config/kafka/kafka-1/server.properties
print(log_dirs(1, ["payments"]))

Équivalent en ligne de commande du `TODO` a (**ne pas lancer en plus du TODO**) :

```bash
# [hôte]
sed -i.bak 's|^log.dirs=/var/lib/kafka/data$|log.dirs=/var/lib/kafka/data,/var/lib/kafka/data2|' \
  config/kafka/kafka-1/server.properties
docker compose restart kafka-1
```

In [ ]:
# TODO 6.5 a : remplacez les deux ... puis exécutez
NEW_LOG_DIRS = ...              # nouvelle valeur de log.dirs : le répertoire actuel, puis /var/lib/kafka/data2,
                                # séparés par une virgule (un texte, sans espace)
PREDICTED_OLD_REFORMATTED = ...  # au redémarrage, le répertoire d'origine /var/lib/kafka/data sera-t-il
                                 # reformaté ? (True ou False)

# Code fourni, à lire : rien à modifier en dessous
format_logs = ""
log_dirs_1 = []
if filled(NEW_LOG_DIRS, PREDICTED_OLD_REFORMATTED):
    if NEW_LOG_DIRS != "/var/lib/kafka/data,/var/lib/kafka/data2":
        # Protection : retirer /var/lib/kafka/data ferait perdre au nœud toutes ses répliques
        print("Valeur refusée : gardez /var/lib/kafka/data en premier, puis ajoutez /var/lib/kafka/data2.")
    else:
        # 1. Copie de sauvegarde (une seule fois), puis remplacement de la ligne log.dirs
        if not os.path.exists(KAFKA_1_FILE + ".bak"):
            shutil.copyfile(KAFKA_1_FILE, KAFKA_1_FILE + ".bak")
        with open(KAFKA_1_FILE) as f:
            content = f.read()
        with open(KAFKA_1_FILE, "w") as f:
            f.write(content.replace("\nlog.dirs=/var/lib/kafka/data\n", "\nlog.dirs=%s\n" % NEW_LOG_DIRS))

        # 2. Configuration statique : redémarrage du nœud, puis attente de son retour
        !docker compose restart kafka-1
        wait_for(lambda: healthy(1) and urp() == 0, timeout=120, message="kafka-1 sain et URP à 0")
        # 3. Traces du démarrage : quels répertoires ont été formatés ?
        format_logs = !docker compose logs kafka-1 --since 2m | grep -E 'start-node|Formatting' 2>&1
        format_logs = format_logs.n
        log_dirs_1 = sorted(log_dirs(1))  # répertoires de données vus par le broker

print(log_dirs_1)

In [ ]:
# Vérification 6.5 a
assert "Formatting data directory /var/lib/kafka/data2" in format_logs, "À revoir : data2 formaté au démarrage"
print("OK  data2 formaté au démarrage")
assert log_dirs_1 == ["/var/lib/kafka/data", "/var/lib/kafka/data2"], "À revoir : kafka-1 a deux répertoires"
print("OK  kafka-1 a deux répertoires")
reformatted = "directory /var/lib/kafka/data with" in format_logs
assert PREDICTED_OLD_REFORMATTED == reformatted, "À revoir : prévision %r : voir les lignes Formatting des traces" % (PREDICTED_OLD_REFORMATTED,)
print("OK  prévision juste : seul data2 est formaté")

**Code fourni** : la partition de `payments` choisie pour le défi, parmi celles dont `kafka-1` héberge une réplique, et la liste de ses répliques dans l'ordre.

In [ ]:
# Code fourni : une partition de payments avec une réplique sur kafka-1
chosen_partition, (_, replicas) = next(((t, p), info) for (t, p), info in sorted(placement(["payments"]).items())
                                       if 1 in info[1])
print("partition :", chosen_partition, "- répliques, dans l'ordre :", replicas)

In [ ]:
# TODO 6.5 b (défi) : remplacez le ... puis exécutez
LOG_DIRS = ...  # la liste log_dirs du plan : un élément par réplique de la liste affichée ci-dessus, dans le même
                # ordre ; "/var/lib/kafka/data2" pour la réplique du broker 1, "any" pour les autres.
                # Exemple pour des répliques [3, 1, 2] : ["any", "/var/lib/kafka/data2", "any"]

# Code fourni, à lire : rien à modifier en dessous
plan_logdir = {}
after_logdir = {}
if filled(LOG_DIRS):
    if len(LOG_DIRS) != len(replicas):
        print("Plan refusé : il faut un élément de LOG_DIRS par réplique, soit %d éléments." % len(replicas))
    else:
        # Le plan d'une seule partition : mêmes répliques, mais un répertoire imposé
        plan_logdir = {"version": 1, "partitions": [{
            "topic": "payments", "partition": chosen_partition[1], "replicas": replicas, "log_dirs": LOG_DIRS}]}
        print(execute(plan_logdir))
        wait_for(lambda: "completed successfully" in verify_plan(), timeout=60, message="déplacement de la réplique")
        after_logdir = log_dirs(1, ["payments"])  # répliques de payments de kafka-1, par répertoire

print(json.dumps(plan_logdir))
print(after_logdir)

In [ ]:
# Vérification 6.5 b
moved = list(after_logdir.get("/var/lib/kafka/data2", {}))
assert len(moved) == 1 and moved[0].startswith("payments-"), "À revoir : une réplique de payments dans data2 : %s" % moved
print("OK  une réplique de payments dans data2 : %s" % moved)
assert not set(moved) & set(after_logdir.get("/var/lib/kafka/data", {})), "À revoir : elle n'est plus dans data"
print("OK  elle n'est plus dans data")

In [ ]:
!docker compose exec -T kafka-1 ls /var/lib/kafka/data2 </dev/null

**Résultat attendu** :

- traces : `Formatting data directory /var/lib/kafka/data2 with metadata.version 4.3-IV0.` ; le répertoire d'origine n'est pas reformaté ;
- `--execute` : `Successfully started moving log directory to /var/lib/kafka/data2 for replica payments-N with broker 1` ;
- `--verify` : `Reassignment of replica payments-N-1 completed successfully.` ;
- la partition apparaît sous `/var/lib/kafka/data2`, et plus sous `/var/lib/kafka/data`.

### 6.6 Rolling restart sous charge

La classe `Load` du TP4 revient : un producteur en arrière-plan, ici `acks=all` et idempotent. Pendant qu'il écrit 90 000 messages à 300 messages/s (5 minutes), redémarrez les nœuds 1 à 3 **un par un**, en attendant entre deux nœuds qu'il soit sain et que l'URP revienne à 0.

In [ ]:
# Code fourni : un producteur de charge qui tourne en arrière-plan (celui du TP4)
class Load:
    """Producteur de charge en arrière-plan : count messages, à rate messages par seconde."""

    def __init__(self, topic, count, rate, config):
        self.confirmed = 0  # messages confirmés par le cluster
        self.errors = []    # erreurs définitives (après toutes les nouvelles tentatives)
        self.producer = Producer({"bootstrap.servers": BS, "log_level": 4, **config})
        # L'envoi tourne dans un fil d'exécution à part : le notebook reste disponible
        self._thread = threading.Thread(target=self._send, args=(topic, count, rate), daemon=True)
        self._thread.start()

    def _report(self, err, msg):
        # Rapport de livraison de chaque message
        if err is None:
            self.confirmed += 1
        else:
            self.errors.append(err)

    def _send(self, topic, count, rate):
        start = time.time()
        for i in range(count):
            # Cadence régulière : le message i part à start + i / rate
            pause = start + i / rate - time.time()
            if pause > 0:
                time.sleep(pause)
            self.producer.produce(topic, value=b"x" * 200, on_delivery=self._report)
            self.producer.poll(0)  # traite les rapports de livraison déjà arrivés
        self.producer.flush(120)   # à la fin, attend les dernières confirmations

    def running(self):
        """Vrai tant que l'envoi n'est pas terminé."""
        return self._thread.is_alive()

    def __repr__(self):
        # Texte affiché par print(load)
        return "Charge(%d confirmés, %d erreurs%s)" % (self.confirmed, len(self.errors),
                                                      ", en cours" if self.running() else ", terminée")

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**), pour chaque nœud `N` :

```bash
# [hôte]
docker compose restart kafka-N
docker compose ps kafka-N        # jusqu'à (healthy)
# [tools]
kafka-topics.sh --bootstrap-server $BS --describe --under-replicated-partitions | wc -l   # jusqu'à 0
```

**Exécutez la cellule `TODO` une seule fois** : elle dure environ 5 minutes.

In [ ]:
# TODO 6.6 : remplacez les deux ... puis exécutez UNE SEULE FOIS (5 minutes environ)
ACTIVE_LAST = ...       # redémarrer le contrôleur actif en dernier ? (True ou False)
PREDICTED_ERRORS = ...  # le producteur de charge (acks=all, idempotent) recevra-t-il des erreurs définitives
                        # pendant le rolling restart ? (True ou False)

# Code fourni, à lire : rien à modifier en dessous
quorum_leader = None
order = []
restart_log = []
load = None
if filled(ACTIVE_LAST, PREDICTED_ERRORS):
    admin = AdminClient(CONF)
    # 1. Le contrôleur actif, noté AVANT le premier redémarrage
    output = !docker exec kafka-tools kafka-metadata-quorum.sh --bootstrap-server $BS describe --status 2>&1
    quorum_leader = int(output.n.split("LeaderId:")[1].split()[0])
    order = [n for n in (1, 2, 3) if n != quorum_leader] + [quorum_leader] if ACTIVE_LAST else [1, 2, 3]

    # 2. La charge : 90000 messages à 300 par seconde, environ 5 minutes
    load = Load("payments", 90000, 300, {"acks": "all", "enable.idempotence": True})
    start = time.time()
    # 3. Un nœud à la fois : redémarrage, puis attente qu'il soit sain et que l'URP revienne à 0
    for n in order:
        !docker compose restart kafka-{n}
        wait_for(lambda: healthy(n) and urp() == 0, timeout=120, message="kafka-%d sain et URP à 0" % n)
        restart_log.append((n, round(time.time() - start), load.confirmed))
    # 4. Fin de la charge, puis leaders préférés remis en tête
    wait_for(lambda: not load.running(), timeout=360, step=5, message="fin de la charge")
    admin.elect_leaders(ElectionType.PREFERRED, None).result()

table(restart_log, ["Nœud", "Sain, URP à 0 (s)", "Messages confirmés"])
print(load)

In [ ]:
# Vérification 6.6
assert sorted(order) == [1, 2, 3] and order[-1] == quorum_leader and len(restart_log) == 3, "À revoir : 3 nœuds redémarrés, contrôleur actif (%s) en dernier : %s" % (quorum_leader, order)
print("OK  3 nœuds redémarrés, contrôleur actif (%s) en dernier : %s" % (quorum_leader, order))
assert load is not None and load.confirmed == 90000 and not load.errors, "À revoir : 90000 messages confirmés, aucune erreur : %s" % load
print("OK  90000 messages confirmés, aucune erreur : %s" % load)
assert urp() == 0, "À revoir : aucune partition sous-répliquée"
print("OK  aucune partition sous-répliquée")
assert PREDICTED_ERRORS == bool(load.errors), "À revoir : prévision %r, %d erreur(s) définitive(s)" % (PREDICTED_ERRORS, len(load.errors))
print("OK  prévision juste : aucune erreur définitive")

**Résultat attendu** :

- le contrôleur actif est redémarré en dernier ;
- chaque nœud redevient sain en 10 à 20 s, et l'URP revient à 0 en quelques secondes ;
- 90 000 messages confirmés, aucune erreur définitive : le producteur a suivi les changements de leader.

### 6.7 Décommissionner kafka-4

Le **cordon** (`cordoned.log.dirs=*`, configuration dynamique du broker) empêche toute nouvelle réplique d'arriver sur le broker. Ensuite : réassigner ses répliques vers les autres brokers, l'arrêter, puis le désenregistrer.

Équivalent en ligne de commande du `TODO` a (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-configs.sh --bootstrap-server $BS --alter --entity-type brokers --entity-name 4 \
  --add-config 'cordoned.log.dirs=*'
```

In [ ]:
# TODO 6.7 a : remplacez les deux ... puis exécutez UNE SEULE FOIS
PREDICTED_CORDON_MOVES = ...  # le cordon déplace-t-il à lui seul les répliques déjà présentes sur le broker 4 ?
                              # (True ou False)
TARGET_BROKERS = ...          # brokers cibles du plan qui vide kafka-4 (une liste de numéros de brokers)

# Code fourni, à lire : rien à modifier en dessous
topics_on_4 = []
plan_brokers = set()
left_on_4 = None
on_4_after_cordon = None
if filled(PREDICTED_CORDON_MOVES, TARGET_BROKERS):
    if 4 in TARGET_BROKERS or len(TARGET_BROKERS) < 3:
        # Protection : le plan doit vider kafka-4 et garder 3 répliques par partition
        print("Plan refusé : il faut 3 brokers cibles au moins, sans le broker 4.")
    else:
        admin = AdminClient(CONF)
        # 1. Le cordon : configuration dynamique du broker 4, plus aucune nouvelle réplique chez lui
        resource = ConfigResource("broker", "4", incremental_configs=[
            ConfigEntry("cordoned.log.dirs", "*", incremental_operation=AlterConfigOpType.SET)])
        for future in admin.incremental_alter_configs([resource]).values():
            future.result()
        on_4_after_cordon = replicas_per_broker().get(4, 0)  # répliques encore sur le broker 4

        # 2. Plan pour tous les topics qui ont une réplique sur le broker 4, vers les brokers cibles
        topics_on_4 = sorted({t for (t, _), (_, replicas) in placement().items() if 4 in replicas})
        rollback_77, plan_77 = generate(topics_on_4, TARGET_BROKERS)
        plan_brokers = {b for p in plan_77["partitions"] for b in p["replicas"]}  # brokers cités dans le plan
        # 3. Exécution limitée à 50 Mo/s, attente de la fin, puis contenu du broker 4
        print(execute(plan_77, 50000000))
        wait_until_done(180)
        left_on_4 = log_dirs(4)

print(topics_on_4, plan_brokers, left_on_4)

In [ ]:
# Vérification 6.7 a
assert plan_brokers == {1, 2, 3}, "À revoir : le plan n'utilise que les brokers 1, 2 et 3"
print("OK  le plan n'utilise que les brokers 1, 2 et 3")
assert left_on_4 == {"/var/lib/kafka/data": {}}, "À revoir : le broker 4 n'héberge plus aucune réplique"
print("OK  le broker 4 n'héberge plus aucune réplique")
assert PREDICTED_CORDON_MOVES == (on_4_after_cordon == 0), "À revoir : prévision %r : %s réplique(s) sur le broker 4 après le cordon" % (PREDICTED_CORDON_MOVES, on_4_after_cordon)
print("OK  prévision juste : %s répliques encore sur le broker 4 après le cordon" % on_4_after_cordon)

Équivalent en ligne de commande du `TODO` b (**ne pas lancer en plus du TODO**) :

```bash
# [hôte]
docker compose --profile scale stop kafka-4
# [tools]
kafka-cluster.sh unregister --bootstrap-server $BS --id 4
```

In [ ]:
# TODO 6.7 b : remplacez les deux ... par vos prévisions, puis exécutez UNE SEULE FOIS
PREDICTED_IN_CLUSTER = ...  # une fois kafka-4 arrêté, et avant unregister, describe_cluster() le montre-t-il
                            # encore parmi les brokers ? (True ou False)
PREDICTED_IN_QUORUM = ...   # juste après l'arrêt, le nœud 4 figure-t-il encore dans describe --replication ?
                            # (True ou False)

# Code fourni, à lire : rien à modifier en dessous
brokers_after_stop = []
role_after_stop = []
unregistration = ""
final_brokers = []
if filled(PREDICTED_IN_CLUSTER, PREDICTED_IN_QUORUM):
    admin = AdminClient(CONF)
    !docker compose --profile scale stop kafka-4
    # 1. Brokers vus par describe_cluster(), et statut du nœud 4 dans le quorum, juste après l'arrêt
    brokers_after_stop = sorted(n.id for n in admin.describe_cluster().result().nodes)
    quorum = !docker exec kafka-tools kafka-metadata-quorum.sh --bootstrap-server $BS describe --replication 2>&1
    print(quorum.n)
    role_after_stop = [line.split()[-1] for line in quorum if line.split()[:1] == ["4"]]
    # 2. Désenregistrement du broker 4, puis liste finale des brokers
    unregistration = !docker exec kafka-tools kafka-cluster.sh unregister --bootstrap-server $BS --id 4 2>&1
    unregistration = unregistration.n
    final_brokers = sorted(n.id for n in admin.describe_cluster().result().nodes)

print(unregistration, final_brokers)

In [ ]:
# Vérification 6.7 b
assert "no longer registered" in unregistration, "À revoir : broker 4 désenregistré"
print("OK  broker 4 désenregistré")
assert final_brokers == [1, 2, 3], "À revoir : brokers 1, 2, 3"
print("OK  brokers 1, 2, 3")
assert not [k for k, (_, r) in placement().items() if 4 in r] and urp() == 0, "À revoir : aucune réplique orpheline, URP à 0"
print("OK  aucune réplique orpheline, URP à 0")
assert PREDICTED_IN_CLUSTER == (4 in brokers_after_stop), "À revoir : prévision %r, brokers après l'arrêt : %s" % (PREDICTED_IN_CLUSTER, brokers_after_stop)
print("OK  prévision juste : brokers après l'arrêt %s" % brokers_after_stop)
assert PREDICTED_IN_QUORUM == bool(role_after_stop), "À revoir : prévision %r, nœud 4 dans le quorum : %s" % (PREDICTED_IN_QUORUM, role_after_stop)
print("OK  prévision juste : nœud 4 encore listé dans le quorum (%s)" % ", ".join(role_after_stop))

**Résultat attendu** :

- topics sur le broker 4 : `orders`, `orders-history`, `payments` ; le plan ne contient que les brokers 1, 2 et 3 ;
- après la réassignation, le répertoire de `kafka-4` est vide ;
- après l'arrêt, le quorum montre encore le nœud 4 en `Observer`, avec un lag qui ne bouge plus ;
- `Broker 4 is no longer registered.` ; restent les brokers 1, 2 et 3, sans partition sous-répliquée.

## Bilan de l'exercice

In [ ]:
# Code fourni : bilan de l'exercice, une ligne par contrôle ; « à compléter » si l'étape n'a pas été faite
table([
    ("kafka-4 enregistré puis retiré", "%s puis %s" % (sorted(brokers), final_brokers), "6.1, 6.7"),
    ("Répliques sur le broker 4 après 6.4", distribution_74.get(4, "à compléter") if distribution_74 else "à compléter", "6.4"),
    ("Répertoires de kafka-1", ", ".join(log_dirs_1) or "à compléter", "6.5"),
    ("Rolling restart : ordre, messages confirmés", "%s, %s" % (order, load), "6.6"),
    ("Partitions sous-répliquées", urp(), "fin"),
], ["Contrôle", "Valeur", "Étape"])

## 7. Questions de réflexion

1. Pourquoi un broker ajouté ne reçoit-il aucune partition existante ? Que se passe-t-il si l'on oublie de réassigner ?
2. Pourquoi conserver la répartition actuelle affichée par `--generate` ? Comment l'utiliseriez-vous ?
3. Que se passe-t-il si l'on oublie de lancer `--verify` jusqu'au bout après un `--throttle` ?
4. Pourquoi kafka-4 est-il `Observer` et non `Follower` dans le quorum ?
5. Une partition vit dans un seul log dir. Quel impact sur l'équilibre entre les disques ? Que se passe-t-il si le disque `data2` tombe en panne ?
6. Pourquoi redémarrer le contrôleur actif en dernier ?
7. Le producteur n'a reçu aucune erreur. Pourquoi ? Qu'est-ce qui aurait changé avec un topic en RF 2 et `min.insync.replicas=2` ?
8. Pourquoi cordonner le broker **avant** de réassigner ? Pourquoi l'arrêter **avant** de le désenregistrer ?